In [ ]:
!pip install segmentation-models-pytorch

In [ ]:
import os
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp


from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

from datasets import load_dataset
from tqdm import tqdm

In [ ]:
#Device 
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
IMG_SIZE = 512

BATCH_SIZE = 4
NUM_WORKERS = 4

NUM_EPOCHS = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5

THRESHOLD = 0.5

MODEL_PATH = "best_unet_vegann.pth"
HISTORY_PATH = "unet_training_history.csv"

print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", NUM_EPOCHS)

In [ ]:
#Verifying VegAnn mask cnversion 
sample = train_hf[0]

image_np = np.array(sample["image"])
mask_np = np.array(sample["mask"])

print("Image shape:", image_np.shape)
print("Mask shape:", mask_np.shape)
print("Mask unique values:", np.unique(mask_np))

In [ ]:
#applying random splits to mask and image 
class VegAnnSegmentationDataset(Dataset):
    def __init__(
        self,
        hf_dataset,
        image_size=512,
        augment=False
    ):
        self.dataset = hf_dataset
        self.image_size = image_size
        self.augment = augment

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]

        image = sample["image"].convert("RGB")
        mask = sample["mask"]

        image = TF.resize(
            image,
            [self.image_size, self.image_size],
            interpolation=InterpolationMode.BILINEAR
        )

        mask = TF.resize(
            mask,
            [self.image_size, self.image_size],
            interpolation=InterpolationMode.NEAREST
        )

        if self.augment:
            if random.random() < 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)

            if random.random() < 0.5:
                image = TF.vflip(image)
                mask = TF.vflip(mask)

        image = TF.to_tensor(image)

        image = TF.normalize(
            image,
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

        mask = np.array(mask)

        # VegAnn foreground/vegetation = 0
        mask = (mask == 0).astype(np.float32)

        mask = torch.from_numpy(mask).unsqueeze(0)

        return {
            "image": image,
            "mask": mask,
            "index": idx
        }

In [ ]:
#creating train and validation datasets 
train_dataset = VegAnnSegmentationDataset(
    train_hf,
    image_size=IMG_SIZE,
    augment=True
)

val_dataset = VegAnnSegmentationDataset(
    val_hf,
    image_size=IMG_SIZE,
    augment=False
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

In [ ]:
#creating Dataloaders 

pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    drop_last=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
#checking one processed batch 
batch = next(iter(train_loader))

print("Image batch:", batch["image"].shape)
print("Mask batch:", batch["mask"].shape)
print("Mask values:", torch.unique(batch["mask"]))

In [ ]:
#Initializing DeepLab ,Model 
model = smp.DeepLabV3(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(device)

total_params = sum(
    p.numel() for p in model.parameters()
)

print("Model: DeepLabV3 + ResNet34")
print("Parameters:", f"{total_params:,}")


In [ ]:
#BCE and Dice loss functions 
dice_loss_fn = smp.losses.DiceLoss(
    mode="binary",
    from_logits=True
)

def combined_loss(logits, targets):
    return dice_loss_fn(logits, targets)

In [ ]:
#mIoU and Dice functions 

def batch_iou_dice(
    logits,
    targets,
    threshold=0.5,
    smooth=1e-6
):
    probabilities = torch.sigmoid(logits)
    predictions = probabilities >= threshold
    targets = targets >= 0.5

    predictions = predictions.flatten(1)
    targets = targets.flatten(1)

    intersection = (
        predictions & targets
    ).sum(dim=1).float()

    union = (
        predictions | targets
    ).sum(dim=1).float()

    pred_sum = predictions.sum(dim=1).float()
    target_sum = targets.sum(dim=1).float()

    iou = (
        intersection + smooth
    ) / (
        union + smooth
    )

    dice = (
        2 * intersection + smooth
    ) / (
        pred_sum + target_sum + smooth
    )

    return iou, dice

In [ ]:
#Optimizer, scheduler and AMP 

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

scheduler = None

"""scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)"""

amp_enabled = torch.cuda.is_available()

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=amp_enabled
)

print("AMP enabled:", amp_enabled)

In [ ]:
#Train one epoch 
def train_one_epoch(model, loader, optimizer, scaler):
    model.train()

    running_loss = 0.0

    progress_bar = tqdm(
        loader,
        desc="Training",
        leave=False
    )

    for batch in progress_bar:
        images = batch["image"].to(
            device,
            non_blocking=True
        )

        masks = batch["mask"].to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp_enabled
        ):
            logits = model(images)
            loss = combined_loss(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    epoch_loss = running_loss / len(loader.dataset)

    return epoch_loss

In [ ]:
#Validation function 
@torch.no_grad()
def validate_model(model, loader):
    model.eval()

    running_loss = 0.0

    all_ious = []
    all_dices = []

    progress_bar = tqdm(
        loader,
        desc="Validation",
        leave=False
    )

    for batch in progress_bar:
        images = batch["image"].to(
            device,
            non_blocking=True
        )

        masks = batch["mask"].to(
            device,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp_enabled
        ):
            logits = model(images)
            loss = combined_loss(logits, masks)

        ious, dices = batch_iou_dice(
            logits,
            masks,
            threshold=THRESHOLD
        )

        running_loss += loss.item() * images.size(0)

        all_ious.extend(
            ious.cpu().numpy().tolist()
        )

        all_dices.extend(
            dices.cpu().numpy().tolist()
        )

    val_loss = running_loss / len(loader.dataset)
    mean_iou = float(np.mean(all_ious))
    mean_dice = float(np.mean(all_dices))

    return val_loss, mean_iou, mean_dice

In [ ]:
#Full training loop 
history = []

best_val_iou = -1.0
training_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scaler
    )

    val_loss, val_iou, val_dice = validate_model(
        model,
        val_loader
    )

    #scheduler.step(val_iou)
    if scheduler is not None:
        scheduler.step(val_iou)

    current_lr = optimizer.param_groups[0]["lr"]
    epoch_time = time.time() - epoch_start

    history.append({
        "Epoch": epoch,
        "Train Loss": train_loss,
        "Validation Loss": val_loss,
        "Validation IoU": val_iou,
        "Validation Dice": val_dice,
        "Learning Rate": current_lr,
        "Epoch Time Seconds": epoch_time
    })

    if val_iou > best_val_iou:
        best_val_iou = val_iou

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_iou": best_val_iou,
                "config": {
                    "image_size": IMG_SIZE,
                    "batch_size": BATCH_SIZE,
                    "epochs": NUM_EPOCHS,
                    "learning_rate": LEARNING_RATE,
                    "seed": SEED
                }
            },
            MODEL_PATH
        )

        saved_message = " — saved best model"
    else:
        saved_message = ""

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"IoU: {val_iou*100:.2f}% | "
        f"Dice: {val_dice*100:.2f}% | "
        f"LR: {current_lr:.2e}"
        f"{saved_message}"
    )

total_training_time = time.time() - training_start

history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)

print()
print(
    "Total training time:",
    f"{total_training_time / 60:.2f} minutes"
)
print(
    "Best validation IoU:",
    f"{best_val_iou*100:.2f}%"
)

In [ ]:
#Plot training and validation loss 
plt.figure(figsize=(9, 5))

plt.plot(
    history_df["Epoch"],
    history_df["Train Loss"],
    marker="o",
    label="Train Loss"
)

plt.plot(
    history_df["Epoch"],
    history_df["Validation Loss"],
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("DeepLabV3+ ResNet34 Training and Validation Loss")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()

plt.savefig(
    "deeplab_loss_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
#plot validaion IoU and Dice 
plt.figure(figsize=(9, 5))

plt.plot(
    history_df["Epoch"],
    history_df["Validation IoU"] * 100,
    marker="o",
    label="Mean IoU"
)

plt.plot(
    history_df["Epoch"],
    history_df["Validation Dice"] * 100,
    marker="o",
    label="Mean Dice"
)

plt.xlabel("Epoch")
plt.ylabel("Score (%)")
plt.title("DeepLabV3 + ResNet34 Validation Performance")
plt.ylim(0, 100)
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()

plt.savefig(
    "deeplab_iou_dice_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
#Loading the best checkpoint 

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Loaded epoch:", checkpoint["epoch"])
print(
    "Best saved IoU:",
    f"{checkpoint['best_val_iou']*100:.2f}%"
)

In [ ]:
@torch.no_grad()
def evaluate_all_metrics(model, loader, threshold=0.5):
    model.eval()

    running_loss = 0.0

    per_image_ious = []
    per_image_dices = []

    total_intersection = 0
    total_union = 0
    total_pred_pixels = 0
    total_gt_pixels = 0

    for batch in tqdm(loader, desc="Final evaluation"):
        images = batch["image"].to(device, non_blocking=True)
        targets = batch["mask"].to(device, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp_enabled
        ):
            logits = model(images)
            loss = combined_loss(logits, targets)

        running_loss += loss.item() * images.size(0)

        predictions = torch.sigmoid(logits) >= threshold
        targets_bool = targets >= 0.5

        pred_flat = predictions.flatten(1)
        target_flat = targets_bool.flatten(1)

        intersections = (pred_flat & target_flat).sum(dim=1).float()
        unions = (pred_flat | target_flat).sum(dim=1).float()

        pred_sums = pred_flat.sum(dim=1).float()
        target_sums = target_flat.sum(dim=1).float()

        image_ious = intersections / unions.clamp_min(1)
        image_dices = (
            2 * intersections
        ) / (
            pred_sums + target_sums
        ).clamp_min(1)

        per_image_ious.extend(image_ious.cpu().tolist())
        per_image_dices.extend(image_dices.cpu().tolist())

        total_intersection += intersections.sum().item()
        total_union += unions.sum().item()
        total_pred_pixels += pred_sums.sum().item()
        total_gt_pixels += target_sums.sum().item()

    val_loss = running_loss / len(loader.dataset)

    mean_per_image_iou = float(np.mean(per_image_ious))
    mean_per_image_dice = float(np.mean(per_image_dices))

    global_iou = (
        total_intersection / total_union
        if total_union > 0 else 0.0
    )

    global_dice = (
        2 * total_intersection
        / (total_pred_pixels + total_gt_pixels)
        if (total_pred_pixels + total_gt_pixels) > 0
        else 0.0
    )

    return {
        "Validation Loss": val_loss,
        "Mean Per-Image IoU": mean_per_image_iou,
        "Mean Per-Image Dice": mean_per_image_dice,
        "Global IoU": global_iou,
        "Global Dice": global_dice
    }

In [ ]:
#Final validation evaluation 
"""final_val_loss, final_mean_iou, final_mean_dice = validate_model(
    model,
    val_loader
)

print(f"Final Validation Loss: {final_val_loss:.4f}")
print(f"Final Mean IoU: {final_mean_iou*100:.2f}%")
print(f"Final Mean Dice: {final_mean_dice*100:.2f}%")"""

final_metrics = evaluate_all_metrics(
    model,
    val_loader,
    threshold=THRESHOLD
)

for metric_name, metric_value in final_metrics.items():
    if metric_name == "Validation Loss":
        print(f"{metric_name}: {metric_value:.4f}")
    else:
        print(f"{metric_name}: {metric_value * 100:.2f}%")


In [ ]:
#Final results table 

unet_results = pd.DataFrame({
    "Method": [
        "DeepLabV3 + ResNet34"
    ],
    "Training Regime": [
        "Fully supervised"
    ],
    "Dataset": [
        "VegAnn"
    ],
    "Train Images": [
        len(train_dataset)
    ],
    "Validation Images": [
        len(val_dataset)
    ],
    "Image Size": [
        IMG_SIZE
    ],
    "Epochs": [
        checkpoint["epoch"]
    ],
    "Parameters": [
        total_params
    ],
    "Mean per Image IoU (%)": [
        final_metrics["Mean Per-Image IoU"] * 100
    ],
    "Mean Dice (%)": [
       final_metrics["Mean Per-Image Dice"] * 100
    ],
    
    "Global IoU (%)": [
        final_metrics["Global IoU"] * 100
    ],

    "Global Dice (%)": [
        final_metrics["Global Dice"] * 100
    ],
    "Training Time (min)": [
        total_training_time / 60
    ]
})

unet_results

In [ ]:
#Saving results 
unet_results.to_csv(
    "deeplab_vegann_results.csv",
    index=False
)

print("Saved DeepLab results.")

In [ ]:
metric_names = [
    "Mean Per-Image IoU",
    "Mean Per-Image Dice",
    "Global IoU",
    "Global Dice"
]

metric_values = [
    final_metrics["Mean Per-Image IoU"] * 100,
    final_metrics["Mean Per-Image Dice"] * 100,
    final_metrics["Global IoU"] * 100,
    final_metrics["Global Dice"] * 100
]

plt.figure(figsize=(10, 6))

bars = plt.bar(
    metric_names,
    metric_values
)

plt.ylabel("Score (%)")
plt.title("DeepLabV3 + ResNet34 Performance on VegAnn Validation Set")
plt.ylim(0, 100)
plt.xticks(rotation=15, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.4)

for bar, value in zip(bars, metric_values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}%",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.tight_layout()

plt.savefig(
    "Deeplab_all_validation_metrics.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
#Prediction helper 
@torch.no_grad()
def predict_single_image(model, dataset, idx):
    model.eval()

    item = dataset[idx]

    image_tensor = item["image"].unsqueeze(0).to(device)
    gt_mask = item["mask"].squeeze(0).numpy().astype(bool)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=amp_enabled
    ):
        logits = model(image_tensor)

    probability = torch.sigmoid(logits)[0, 0]
    pred_mask = (
        probability >= THRESHOLD
    ).cpu().numpy()

    original_sample = val_hf[idx]
    original_image = np.array(
        original_sample["image"].convert("RGB")
    )

    original_image = np.array(
        Image.fromarray(original_image).resize(
            (IMG_SIZE, IMG_SIZE)
        )
    )

    intersection = np.logical_and(
        pred_mask,
        gt_mask
    ).sum()

    union = np.logical_or(
        pred_mask,
        gt_mask
    ).sum()

    iou = (
        intersection / union
        if union > 0
        else 0.0
    )

    dice_denominator = (
        pred_mask.sum() + gt_mask.sum()
    )

    dice = (
        2 * intersection / dice_denominator
        if dice_denominator > 0
        else 0.0
    )

    return (
        original_image,
        gt_mask,
        pred_mask,
        iou,
        dice
    )

In [ ]:
#Visualize 
visual_idx = 0

image_vis, gt_vis, pred_vis, sample_iou, sample_dice = (
    predict_single_image(
        model,
        val_dataset,
        visual_idx
    )
)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(image_vis)
plt.title("Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(gt_vis, cmap="gray")
plt.title("Ground Truth Mask")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(pred_vis, cmap="gray")
plt.title(
    f"DeepLabV3+ ResNet34 Prediction\n"
    f"IoU={sample_iou*100:.2f}%, "
    f"Dice={sample_dice*100:.2f}%"
)
plt.axis("off")

plt.tight_layout()

plt.savefig(
    "deeplab_image_gt_prediction.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
#Find best median and worst validation examples 
@torch.no_grad()
def collect_validation_predictions(model, dataset):
    records = []

    for idx in tqdm(
        range(len(dataset)),
        desc="Collecting predictions"
    ):
        _, _, _, iou, dice = predict_single_image(
            model,
            dataset,
            idx
        )

        records.append({
            "Index": idx,
            "IoU": iou,
            "Dice": dice
        })

    return pd.DataFrame(records)


prediction_records = collect_validation_predictions(
    model,
    val_dataset
)

prediction_records = prediction_records.sort_values(
    "IoU"
).reset_index(drop=True)

worst_idx = int(
    prediction_records.iloc[0]["Index"]
)

median_idx = int(
    prediction_records.iloc[
        len(prediction_records) // 2
    ]["Index"]
)

best_idx = int(
    prediction_records.iloc[-1]["Index"]
)

print("Worst index:", worst_idx)
print("Median index:", median_idx)
print("Best index:", best_idx)

In [ ]:
#Visualize all 3 
example_indices = [
    worst_idx,
    median_idx,
    best_idx
]

example_names = [
    "Worst",
    "Median",
    "Best"
]

fig, axes = plt.subplots(
    3,
    3,
    figsize=(15, 15)
)

for row, (idx, name) in enumerate(
    zip(example_indices, example_names)
):
    image_vis, gt_vis, pred_vis, iou, dice = (
        predict_single_image(
            model,
            val_dataset,
            idx
        )
    )

    axes[row, 0].imshow(image_vis)
    axes[row, 0].set_title(f"{name}: Image")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(
        gt_vis,
        cmap="gray"
    )
    axes[row, 1].set_title("Ground Truth")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(
        pred_vis,
        cmap="gray"
    )
    axes[row, 2].set_title(
        f"Prediction\n"
        f"IoU={iou*100:.2f}%, "
        f"Dice={dice*100:.2f}%"
    )
    axes[row, 2].axis("off")

plt.tight_layout()

plt.savefig(
    "deeplab_best_median_worst.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
unet_result_row = pd.DataFrame({
    "Method": ["U-Net + ResNet34"],
    "Training Regime": ["Fully supervised"],
    "Epochs": [15],
    "Validation Images": [len(val_dataset)],
    "Mean Per-Image IoU (%)": [
        all_metrics["Mean Per-Image IoU"] * 100
    ],
    "Mean Per-Image Dice (%)": [
        all_metrics["Mean Per-Image Dice"] * 100
    ],
    "Global IoU (%)": [
        all_metrics["Global IoU"] * 100
    ],
    "Global Dice (%)": [
        all_metrics["Global Dice"] * 100
    ]
})

unet_result_row